In [ ]:
# ??
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']  # ?????????
plt.rcParams['axes.unicode_minus'] = False  # ??????
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
import joblib
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
error_list=['????????1001',
'????????2001',
'????????4001',
'????????4002',
'????????4003',
'????????5001',
'????????5002',
'????????6001',
'????????6002'
]
error_value=[1001, 2001, 4001, 4002, 4003, 5001, 5002, 6001, 6002]

In [ ]:
def model(X,y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True)
    # ????
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    # ??????????
    model1 = RandomForestClassifier(n_estimators=100, random_state=42)  # n_estimators?????
    model2 = GradientBoostingClassifier(
        n_estimators=200,  # ????(?)???
        learning_rate=0.1,  # ??????????
        max_depth=15,  # ????????
        min_samples_split=50,  # ??????????????
        min_samples_leaf=10,  # ?????????????
        subsample=0.6,  # ??????????
        random_state=42  # ???????????
        )
    model3 = XGBClassifier(random_state=120,
                      objective = 'multi:softmax',
                      eval_metric = ['mlogloss', 'aucpr', 'auc', 'map'],
                      num_class=2,
                      n_estimators=340,
                      reg_lambda=2,
                      min_child_weight=0,
                      max_depth=6,
                      learning_rate=0.1)
    # ????????
    model4 = LogisticRegression(max_iter=1500, penalty='l2', solver='lbfgs')
    # ????????????
    ensemble = VotingClassifier(estimators=[
        ('RF', model1),
        ('GBDT', model2),
        ('XGB', model3),
        ('LR',model4)
    ], voting='hard')

    # ????????????
    ensemble.fit(X_train, y_train)

    # ???????????????
    print(f"????:{ensemble.score(X_test, y_test) * 100:.2f}%")

    # ????????????????
    y_pred = ensemble.predict(X_test)
    conf_matrix = confusion_matrix(y_test, y_pred)

    print("????:")
    print(conf_matrix)

    # ??????,?????????????
    print("\n????:")
    print(classification_report(y_test, y_pred))
    
    # ????
#     scaler = StandardScaler()
#     X = scaler.fit_transform(X)
#     # ????????????
#     ensemble.fit(X, y)

    ensemble_name = error_list[i]+'.pkl'
    # ?????????????
    joblib.dump(ensemble, ensemble_name)

    # ??????
    print(f"???????????? {ensemble_name}")


In [ ]:
for i in range(len(error_list)):
    df = pd.DataFrame()
    df_name = './data/error/'+ str(error_value[i])+ '.csv'
    df = pd.read_csv(df_name, index_col=0)
    n = len(df.columns)
    # ???????
    dfx = df.iloc[:, :n-9]
    # ????????
    dfx = dfx.drop(['?????'], axis=1)
    # ?????????? 0 ????? 1
    df[error_list[i]] = (df[error_list[i]] != 0).astype(int)
    X=dfx
    y=df[error_list[i]]
    print(y.value_counts())
    model(X,y)
    print()